# 05 Evalute Baseline

Evaluate 40 examples across three systems: pretrained `flan-t5-base`, prompt-engineered `flan-t5-base`, and RAG-grounded `flan-t5-base`. The notebook writes metrics, generations, and a markdown comparison report.

## Setup

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Could not locate project root containing src/')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.baseline_eval import run_baseline_evaluation

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
pd.set_option('display.max_colwidth', 180)
print(f'Project root: {PROJECT_ROOT}')

## Run Baseline Evaluation

This cell runs 40 test examples. It may take several minutes on a MacBook because it generates outputs for three strategies.

In [ ]:
metrics_df, aggregate_df, generations_df = run_baseline_evaluation(
    finetune_test_path=DATA_DIR / 'finetune_test.jsonl',
    chroma_dir=DATA_DIR / 'chroma',
    output_dir=OUTPUTS_DIR,
    sample_size=40,
)

print(f'Metric rows: {len(metrics_df)}')
print(f'Generation rows: {len(generations_df)}')

## Aggregate Comparison Table

In [ ]:
aggregate_df

## Task-Level Metrics

In [ ]:
task_metrics = (
    metrics_df.groupby(['model', 'task'])[['bleu', 'rouge1', 'rouge2', 'rougeL', 'bertscore_f1']]
    .mean()
    .reset_index()
    .sort_values(['task', 'model'])
)
task_metrics

## Output Files

In [ ]:
artifact_paths = [
    OUTPUTS_DIR / 'baseline_40_metrics.csv',
    OUTPUTS_DIR / 'baseline_40_generations.csv',
    OUTPUTS_DIR / 'baseline_40_comparison_table.csv',
    OUTPUTS_DIR / 'baseline_40_comparison.md',
]
pd.DataFrame(
    {
        'artifact': [str(path.relative_to(PROJECT_ROOT)) for path in artifact_paths],
        'exists': [path.exists() for path in artifact_paths],
        'size_bytes': [path.stat().st_size if path.exists() else 0 for path in artifact_paths],
    }
)

## Sample Generations

In [ ]:
generations_df[['example_id', 'strategy', 'task', 'topic', 'candidate']].head(12)